# `json2vec` Hello World

This notebook trains the smallest useful JSON2Vec model: two numeric Iris measurements predict the Iris species. The point is not accuracy. The point is to show the complete loop from records, to schema, to training, to prediction and embeddings. This example is intentionally flat; the next tutorials show why arrays matter.

Start with the normal training dependencies plus the bundled Iris dataset. The examples remove notebook logging noise so the rendered docs stay focused on model behavior.

In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint
from sklearn.datasets import load_iris

import json2vec as j2v

logger.remove()

Build a tiny Polars frame from a balanced slice of Iris rows. The schema field names match the DataFrame columns, so JSON2Vec can infer the request queries.

In [2]:
iris = load_iris()
indices = [*range(0, 12), *range(50, 62), *range(100, 112)]

records = pl.DataFrame(
    {
        "sepal_length": iris.data[indices, 0].tolist(),
        "petal_length": iris.data[indices, 2].tolist(),
        "species": [iris.target_names[index] for index in iris.target[indices]],
    }
)

records.head()

sepal_length,petal_length,species
f64,f64,str
5.1,1.4,"""setosa"""
4.9,1.4,"""setosa"""
4.7,1.3,"""setosa"""
4.6,1.5,"""setosa"""
5.0,1.4,"""setosa"""


The schema declares exactly what the model should read. `Number` fields become numeric tensorfields, and the `Category` field is a supervised target because `target=True` hides it from the input and asks the model to decode it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("sepal_length"),
    j2v.Number("petal_length"),
    j2v.Category("species", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)

Train for one deliberately small epoch. The tutorials keep batch and epoch counts hardcoded so the example remains quick to run in documentation builds.

In [4]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

Prediction uses the same nested batch shape as training: each outer item is one observation, and each observation contains one record.

In [5]:
batch = [[record] for record in records.to_dicts()[:3]]

In [6]:
pprint(model.predict(batch))

{
│   'record/species': {
│   │   'state': {
│   │   │   'valued': [0.5611112117767334, 0.5621271729469299, 0.5617240071296692],
│   │   │   'null': [0.07378394901752472, 0.07275740802288055, 0.07336356490850449],
│   │   │   'padded': [0.07353905588388443, 0.07413437217473984, 0.07477101683616638],
│   │   │   'masked': [0.22962439060211182, 0.22914519906044006, 0.2273353934288025],
│   │   │   'other': [0.06194165721535683, 0.061835877597332, 0.06280609965324402]
│   │   },
│   │   'content': {
│   │   │   'value': ['setosa', 'setosa', 'setosa'],
│   │   │   'probability': [0.4080601930618286, 0.40151938796043396, 0.4067239761352539],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.4080601930618286},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.3837924003601074}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.40151938796043396},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.38732293248176575}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.4067239761352539},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.3838869333267212}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

Embeddings are opt-in. Mutating the root `record` node with `embed=True` exposes a vector for each input observation without changing the schema fields themselves.

In [7]:
model.set(j2v.where("name") == "record", embed=True)

pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.0804007425904274,
│   │   │   │   -0.27828243374824524,
│   │   │   │   0.12410859763622284,
│   │   │   │   -0.16616646945476532,
│   │   │   │   -0.19717994332313538,
│   │   │   │   0.09851796925067902,
│   │   │   │   -0.08945125341415405,
│   │   │   │   -0.08754283934831619,
│   │   │   │   -0.08304978907108307,
│   │   │   │   -0.03324655070900917,
│   │   │   │   -0.005446013528853655,
│   │   │   │   -0.27175015211105347,
│   │   │   │   -0.12116469442844391,
│   │   │   │   0.5091058015823364,
│   │   │   │   0.6623668074607849,
│   │   │   │   -0.11978985369205475
│   │   │   ],
│   │   │   [
│   │   │   │   0.06469862163066864,
│   │   │   │   -0.2634967565536499,
│   │   │   │   0.11460870504379272,
│   │   │   │   -0.154901921749115,
│   │   │   │   -0.18989895284175873,
│   │   │   │   0.07638036459684372,
│   │   │   │   -0.05347345396876335,
│   │   │   │   -0.10287930071353912,
│   │   │   │   -0.08318933099508286,
│   │   │   │   -0.03661907836794853,
│   │   │   │   -0.029736053198575974,
│   │   │   │   -0.2803056538105011,
│   │   │   │   -0.12188920378684998,
│   │   │   │   0.5231586694717407,
│   │   │   │   0.6677305698394775,
│   │   │   │   -0.10854316502809525
│   │   │   ],
│   │   │   [
│   │   │   │   0.07570286840200424,
│   │   │   │   -0.2714507281780243,
│   │   │   │   0.1095840334892273,
│   │   │   │   -0.14827454090118408,
│   │   │   │   -0.19348730146884918,
│   │   │   │   0.09259040653705597,
│   │   │   │   -0.06415707617998123,
│   │   │   │   -0.08471208065748215,
│   │   │   │   -0.08953754603862762,
│   │   │   │   -0.052980247884988785,
│   │   │   │   0.0005079496768303216,
│   │   │   │   -0.27989399433135986,
│   │   │   │   -0.12923890352249146,
│   │   │   │   0.5254405736923218,
│   │   │   │   0.6570091843605042,
│   │   │   │   -0.12518157064914703
│   │   │   ]
│   │   ]
│   }
}

The plot is the quickest way to verify what was built: array nodes, tensorfield nodes, targets, embeddings, and parameter counts all appear in the same tree.

In [8]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  18,153                                │
│  Schema map                                                            arrays  1                                     │
│                                                                        fields  3                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               